# SymGate-VDSR inference + GeoTIFF export

This notebook runs the **SymGateVDSR** model on scene-level `.npz` files with keys:

- `tensor`: `(4, H, W)`
  - Ch1 = baseline / bicubic DEM
  - Ch2 = raw ICESat-2 / LiDAR elevations
  - Ch3 = photon mask
  - Ch4 = GT DEM (ignored at inference)
- `geotransform`: GDAL-style `(x0, dx, rx, y0, ry, -dy)`
- `epsg`: integer CRS code

It will:

1. load a wrapped training checkpoint (`model_state_dict`)
2. tile each variable-sized scene into overlapping windows
3. run model inference
4. stitch windows with **HannStreamMerger**
5. write a georeferenced **GeoTIFF** that opens in **ArcGIS** and **QGIS**

> Put this notebook in the same folder as:
>
> - `model_symgatevdsr.py`
> - `hann_merge.py`
> - `symgate_infer_export.py`


In [1]:
from pathlib import Path
import sys

# Make local helper files importable if needed
HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from symgate_infer_export import load_model, run_one, collect_npz_files


In [2]:
# ---------------------------
# CONFIGURATION
# ---------------------------
CHECKPOINT_PATH = Path(r"D:\Vaibhav\dem-lidar\checkpoints_symgate_vdsrold/symgate_vdsr_best.pt")
INPUT_PATH      = Path(r"D:\Vaibhav\dem-lidar\tensors_256\train")
OUTPUT_DIR      = Path(r"D:\Vaibhav\dem-lidar\out_symgate_vdsr")

DEVICE      = "cuda"   # change to "cpu" if needed
PATCH_SIZE  = 256
OVERLAP     = 192
PAD         = None      # None -> patch_size // 2
BATCH_SIZE  = 16
USE_AMP     = True      # mixed precision on CUDA
STRICT_LOAD = True     # wrapped checkpoints often load fine with False
SKIP_EXISTING = True


In [3]:
model = load_model(
    checkpoint_path=CHECKPOINT_PATH,
    device=DEVICE,
    strict=STRICT_LOAD,
)
print("Model loaded.")


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL torch.torch_version.TorchVersion was not an allowed global by default. Please use `torch.serialization.add_safe_globals([torch.torch_version.TorchVersion])` or the `torch.serialization.safe_globals([torch.torch_version.TorchVersion])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load 

In [ ]:
# Preview which files will run
npz_files = collect_npz_files(INPUT_PATH)
print(f"Found {len(npz_files)} npz files")
npz_files[:5]


In [ ]:
# ---------------------------
# RUN ONE FILE
# ---------------------------
# sample_npz = npz_files[0]
# out_path = run_one(
#     npz_path=sample_npz,
#     model=model,
#     output_dir=OUTPUT_DIR,
#     patch_size=PATCH_SIZE,
#     overlap=OVERLAP,
#     pad=PAD,
#     batch_size=BATCH_SIZE,
#     device=DEVICE,
#     amp=USE_AMP,
#     skip_existing=False,
# )
# print(out_path)


In [ ]:
# ---------------------------
# RUN WHOLE FOLDER
# ---------------------------
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
written = []
for npz_path in npz_files:
    out_path = run_one(
        npz_path=npz_path,
        model=model,
        output_dir=OUTPUT_DIR,
        patch_size=PATCH_SIZE,
        overlap=OVERLAP,
        pad=PAD,
        batch_size=BATCH_SIZE,
        device=DEVICE,
        amp=USE_AMP,
        skip_existing=SKIP_EXISTING,
    )
    written.append(out_path)

print(f"Done. Wrote {len(written)} GeoTIFF(s) to {OUTPUT_DIR}")


## CLI alternative

If you prefer terminal use, run:

```bash
python symgate_infer_export.py   --checkpoint "D:/path/to/checkpoint.pt"   --input "D:/Vaibhav_00006/dataset/dataset_15m/tensors/train"   --output-dir "D:/Vaibhav_00006/dataset/dataset_15m/predictions_symgate"   --device cuda   --patch-size 256   --overlap 192   --batch-size 16   --amp   --skip-existing
```
